# 01 · Scene basics: conventions and usage

How OptiVolume describes a refractive-index scene: units and axes, shapes and their soft edges,
refractive index (constant or varying inside a shape), how shapes combine, how rotations work, and
voxel objects. Everything is differentiable, so every parameter can be fitted.

| convention | rule |
|---|---|
| units | SI: metres for lengths, degrees for rotation angles in constructors |
| grid | sample `(i, j, k)` sits at `(i·dx, j·dy, k·dz)`; arrays are indexed `[x, y, z]` |
| shapes | signed distance (negative inside); occupancy `sigmoid(−sdf/softness)` |
| softness | metres; default `None` = set by the volume, a 10–90 % edge about one voxel wide |
| refractive index | separate from geometry: a scalar, or a map in the shape's own frame |
| composition | in the order added; a later shape paints over earlier ones |
| rotation | the object is rotated by R about its centre; right-handed (+90° about z turns +x into +y) |
| sampling | backward map: each grid point is mapped into the object's frame (no holes) |
| voxel objects | trilinear interpolation; outside their box they are the background `n0` |

In [ ]:
import torch
from matplotlib import pyplot as plt

from optical_volume import scene

N, D, N_BG = 64, 0.1e-6, 1.33                 # 64³ samples of 100 nm, water background
C = torch.full((3,), (N - 1) * D / 2)         # the grid centre, in metres


def show_xy(ri, k=N // 2, ax=None, title=""):
    """Plot the slice z = k with x to the right and y up (arrays are indexed [x, y, z])."""
    ax = ax or plt.gca()
    ext = [0, N * D * 1e6, 0, N * D * 1e6]
    im = ax.imshow(ri[:, :, k].detach().T, origin="lower", extent=ext, cmap="viridis")
    ax.set_xlabel("x (µm)"); ax.set_ylabel("y (µm)"); ax.set_title(title, fontsize=9)
    return im

## 1. Shapes are signed distances with a soft edge

A shape knows its signed distance to every point: negative inside, positive outside, in metres.
Its **occupancy** is `sigmoid(−sdf / softness)`: 1 inside, 0 outside, exactly 0.5 on the surface.
`softness` is in metres, and the 10–90 % edge width is `4.39 × softness` for every shape and size.
Leave it at `None` and the volume chooses about one voxel. That keeps the geometry differentiable: a
hard step would have no gradient with respect to centre or size.

In [ ]:
sphere = scene.shapes.Sphere(C.clone(), torch.tensor(2e-6), torch.tensor(1.46))
x = torch.linspace(1.0e-6, 3.0e-6, 401)
pts = torch.stack([C[0] + x, C[1].expand_as(x), C[2].expand_as(x)], dim=-1)
with torch.no_grad():
    for s in (10e-9, D / 4.39, 100e-9):
        plt.plot(x * 1e6, sphere(pts, softness=s), label=f"softness {s * 1e9:.0f} nm")
plt.axvline(2.0, color="0.6", ls=":"); plt.axhline(0.5, color="0.8", lw=0.6)
plt.xlabel("distance from the centre (µm)"); plt.ylabel("occupancy"); plt.legend(fontsize=8)
plt.title("occupancy across the surface of a 2 µm sphere", fontsize=9);

## 2. Refractive index: a separate feature, scalar or map

The refractive index is not part of the geometry. Give a shape a number (a trainable parameter
`RI`), or a **map** that varies inside it:
- `scene.VoxelRI(values, spacing)`: a trainable voxel grid centred on the shape;
- `scene.FunctionRI(fn)`, or any function or `nn.Module` of the shape's own coordinates
  (metres, relative to its centre, rotated with it).

`CollectionVolume` combines the shapes in the order they were added:
`n ← n·(1 − occupancy) + RI·occupancy`, so a later shape paints over earlier ones.

In [ ]:
vol = scene.CollectionVolume((N, N, N), (D, D, D), N_BG)
graded = scene.shapes.Ellipsoid(C - torch.tensor([1.2e-6, 0, 0]), torch.tensor([1.6e-6, 2.4e-6, 1.6e-6]),
                                lambda p: 1.40 + 3e4 * p[..., 1])            # +0.03 per µm along its own y
cube = scene.shapes.Cube(C + torch.tensor([1.6e-6, 0, 0]), torch.tensor(1.6e-6), torch.tensor(1.50))
core = scene.shapes.Sphere(C + torch.tensor([1.6e-6, 0, 0]), torch.tensor(0.5e-6), torch.tensor(1.38))
vol.add([graded, cube, core])                      # the core is added last, so it sits inside the cube
with torch.no_grad():
    im = show_xy(vol(), title="graded ellipsoid, cube with a core added after it")
plt.colorbar(im, label="refractive index");

## 3. Rotations

Every shape has a centre and a rotation (a quaternion `quat`, set from an axis and an angle in
degrees). The convention:
- **Active:** the object is rotated by R about its centre.
- **Right-handed:** +90° about z turns +x into +y.
- **Backward map:** to draw it, each grid point g is mapped into the object's frame,
  p = Rᵀ(g − c), and the object is evaluated there. Every grid point gets one value, so a rotated
  object has no holes.

`set_quat(axis, angle)` changes the orientation in place, so an optimizer holding `quat` keeps
working. Shapes and `VoxelVolume` follow the same convention. Below, a bar along x turned about
z: at +30° its +x end has moved towards +y, and at +90° it lies along y.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
bar = scene.shapes.Cube(C.clone(), torch.tensor([4e-6, 1e-6, 1e-6]), torch.tensor(1.46))
for a, angle in zip(ax, (0, 30, 90)):
    v = scene.CollectionVolume((N, N, N), (D, D, D), N_BG)
    bar.set_quat((0, 0, 1), angle)
    v.add([bar])
    with torch.no_grad():
        show_xy(v(), ax=a, title=f"bar rotated {angle}° about z")
plt.tight_layout()

## 4. Voxel objects

`VoxelVolume` holds an object as a voxel array (for example from a mesh or a measurement) and
samples it onto any grid with trilinear interpolation, rotated by the same convention. Outside its
own box it is exactly the background `n0`. Below, a voxelized copy of the bar, rotated by 30° in
`VoxelVolume`, matches the analytic bar rotated by 30°.

In [ ]:
grid = scene.volume.Grid((N, N, N), (D, D, D)).grid
v = scene.CollectionVolume((N, N, N), (D, D, D), N_BG)
bar.set_quat((0, 0, 1), 0); v.add([bar])
with torch.no_grad():
    voxels = v()                                            # the unrotated bar as voxels
    vv = scene.VoxelVolume((N, N, N), (D, D, D), voxels, n0=N_BG, rotation_axis=(0, 0, 1), rotation_angle=30.)
    from_voxels = vv(grid)
    bar.set_quat((0, 0, 1), 30)
    analytic = v()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
show_xy(from_voxels, ax=ax[0], title="VoxelVolume, 30° about z")
show_xy(analytic, ax=ax[1], title="analytic Cube, 30° about z")
im = ax[2].imshow((from_voxels - analytic)[:, :, N // 2].T, origin="lower", cmap="RdBu_r", vmin=-0.05, vmax=0.05,
                  extent=[0, N * D * 1e6, 0, N * D * 1e6])
ax[2].set_title("difference (interpolation at the edges)", fontsize=9); fig.colorbar(im, ax=ax[2])
plt.tight_layout()

## 5. Slabs

`Slab(point, normal, thickness, RI)` spans `|n·(x − point)| ≤ thickness/2`: a coverslip, a flat
layer, a tilted interface. It is oriented by its normal (any length); its own z axis points
along the normal, so an RI map inside a slab varies across it.

In [ ]:
v = scene.CollectionVolume((N, N, N), (D, D, D), N_BG)
v.add([scene.shapes.Slab(C.clone(), torch.tensor([1.0, 0.5, 0.0]), torch.tensor(1.5e-6), 1.52)])
with torch.no_grad():
    show_xy(v(), title="slab, normal (1, 0.5, 0), 1.5 µm thick");

## 6. Everything is a parameter

Centres, sizes, rotations, refractive indices and RI maps are `nn.Parameter`s. Any loss on the
volume (or on an image simulated from it) has gradients with respect to all of them. Freeze the
ones you don't want to fit with `requires_grad_(False)`.

In [ ]:
v = scene.CollectionVolume((N, N, N), (D, D, D), N_BG)
s = scene.shapes.Sphere(C.clone(), torch.tensor(1.5e-6), torch.tensor(1.45))
v.add([s])
target_radius = 1.8e-6
with torch.no_grad():
    target = scene.CollectionVolume((N, N, N), (D, D, D), N_BG)
    target.add([scene.shapes.Sphere(C.clone(), torch.tensor(target_radius), torch.tensor(1.45))])
    target = target()
s.RI.requires_grad_(False)                                 # fit only centre and radius
opt = torch.optim.Adam(s.parameters(), lr=2e-8)
for step in range(100):
    opt.zero_grad()
    loss = ((v() - target) ** 2).mean()
    loss.backward()
    opt.step()
print(f"radius {s.radius.item() * 1e6:.3f} µm (target {target_radius * 1e6:.1f} µm), loss {loss.item():.2e}")